# Implements a random Gx/Gx/c queue generator
### Authors: Rafael Andrade & Eriky Silva & Frederico Cruz
---

Required packages

In [84]:
import random as rnd
import heapq as hq
import pandas as pd
import numpy as np
import math
import scipy.optimize as opt

import pickle
from datetime import date

## Queue generator class
---

In [85]:
class Queue:
    def __init__(self, farr, fdep, tmax, nserv=1, farr_batch=lambda: 1, fdep_batch=lambda: 1):
        self.elements = []
        self.nserv = nserv
        self.farr = farr
        self.fdep = fdep
        self.farr_batch = farr_batch
        self.fdep_batch = fdep_batch
        self.tmax = tmax

        self.queue = 0
        self.inserv = 0
        self.busyserv = 0
        self.time = 0
        self.events = []

        self.log_time = []
        self.log_event = []
        self.log_queue = []
        self.log_inserv = []
        self.log_busy = []
        self.total_arrivals = 0
        self.total_departures = 0
        
        self.schedule_arr()


    def schedule_arr(self):
        arr_time = self.time + self.farr()
        hq.heappush(self.events, (arr_time, 'arr', 1))


    def schedule_dep(self, batch):
        dep_time = self.time + self.fdep()
        hq.heappush(self.events, (dep_time, 'dep', batch))


    def try_start_service(self):
        while self.busyserv < self.nserv and self.queue > 0:
            batch = min(self.fdep_batch(), self.queue)
            self.queue -= batch
            self.inserv += batch
            self.busyserv += 1
            self.schedule_dep(batch)

            
    def process_arr(self):
        self.schedule_arr()
        batch = self.farr_batch()
        self.queue += batch
        self.total_arrivals += batch
        self.try_start_service()


    def process_dep(self, batch):
        self.busyserv -= 1
        self.inserv -= batch
        self.total_departures += batch
        self.try_start_service()


    def log_state(self, event):
        self.log_time.append(self.time)
        self.log_event.append(event)
        self.log_queue.append(self.queue)
        self.log_inserv.append(self.inserv)
        self.log_busy.append(self.busyserv)


    def run(self):
        while self.events and self.time < self.tmax:
            self.time, ev, batch = hq.heappop(self.events)

            if ev == 'arr':
                self.process_arr()
            else:
                self.process_dep(batch)

            self.log_state(ev)

    
    def get_metrics(self):
        if len(self.log_time) < 2:
            print("Not enough data")
            return

        total_time = self.log_time[-1]

        area_q = 0.0
        area_inserv = 0.0
        area_b = 0.0
        area_s = 0.0

        for i in range(1, len(self.log_time)):
            dt = self.log_time[i] - self.log_time[i-1]

            q = self.log_queue[i-1]
            inserv = self.log_inserv[i-1]
            b = self.log_busy[i-1]
            s = q + inserv

            area_q += q * dt
            area_inserv += inserv * dt
            area_b += b * dt
            area_s += s * dt

        avg_q = area_q / total_time
        avg_inserv = area_inserv / total_time
        avg_b = area_b / total_time
        avg_s = area_s / total_time

        n_arr = self.total_arrivals
        n_dep = self.total_departures

        utilization = avg_b / self.nserv
        lmbda_eff = n_arr / total_time
        throughput = n_dep / total_time

        w = avg_s / lmbda_eff if lmbda_eff > 0 else 0
        wq = avg_q / lmbda_eff if lmbda_eff > 0 else 0

        return {
        "rho": utilization,
        "L": avg_s,
        "Lq": avg_q,
        "W": w,
        "Wq": wq,
        "avg_in_service": avg_inserv,
        "throughput": throughput,
        "total_time": total_time,
        "arrivals": n_arr,
        "departures": n_dep
    }

    def print_metrics(self):
        metrics = self.get_metrics()
        print("=== Queue Metrics ===")
        print(f"Rho: {metrics['rho']:.4f}")
        print(f"L: {metrics['L']:.4f}")
        print(f"Lq: {metrics['Lq']:.4f}")
        print(f"W: {metrics['W']:.4f}")
        print(f"Wq: {metrics['Wq']:.4f}")
        print(f"Avg in service: {metrics['avg_in_service']:.4f}")
        print(f"Throughput: {metrics['throughput']:.4f}")
        print(f"Total time: {metrics['total_time']:.4f}")
        print(f"Arrivals: {metrics['arrivals']}")
        print(f"Departures: {metrics['departures']}")

    
    def print_log(self):
        return pd.DataFrame({
            'time': self.log_time,
            'event': self.log_event,
            'queue': self.log_queue,
            'inserv': self.log_inserv,
            'busy': self.log_busy
        })


## Simulating queues
---

#### Monte Carlo functions

In [86]:
def mc_queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch, rep_mc=100):
    np.random.seed(2026)
    res = []
    for _ in range(rep_mc):
        q = Queue(farr, fdep, tmax, nserv, farr_batch, fdep_batch)
        q.run()
        m = q.get_metrics()
        res.append([m['Wq'], m['Lq'], m['rho']])
    
    df = pd.DataFrame(res, columns=['Wq', 'Lq', 'Rho'])
    return df.mean().add_suffix('_mean').to_dict() | df.var().add_suffix('_var').to_dict()


def tab_mc_queue(lambdas, mus, tmaxs, farr, fdep, nserv, farr_batch, fdep_batch, rep_mc=100):
    tab = []
    total = len(tmaxs) * len(lambdas)
    k = 0
    for lmb, mu in zip(lambdas, mus):
        for tmax in tmaxs:
            k += 1
            stats = mc_queue(farr(lmb), fdep(mu), tmax, nserv, farr_batch, fdep_batch, rep_mc)
            tab.append({'lambda': lmb, 'mu': mu, 'tmax': tmax} | stats)
            print(f"\r{100*k/total:.1f}%", end="")
    return pd.DataFrame(tab)

#### Theoretical formulas

In [ ]:
def mmc_metrics(lmbda, mu, c):
    rho = lmbda / (c * mu)
    if rho >= 1:
        return None  # System is unstable

    sum_terms = sum((lmbda / mu) ** n / math.factorial(n) for n in range(c))
    p0 = 1 / (sum_terms + (lmbda / mu) ** c / (math.factorial(c) * (1 - rho)))
    lq = (p0 * (lmbda / mu) ** c * rho) / (math.factorial(c) * (1 - rho) ** 2)
    wq = lq / lmbda

    return {
        "Rho": rho,
        "Lq": lq,
        "Wq": wq
    }

def mxm1_metrics(lmbda, mu, batch):
    k = batch
    rho = (lmbda * k) / mu
    
    if rho >= 1:
        return None  # System is unstable
        
    l = (rho + (lmbda / mu) * (k**2)) / (2 * (1 - rho))
    lq = l - rho
    wq = lq / (lmbda * k)
    
    return {
        "Rho": rho,
        "Lq": lq,
        "Wq": wq
    }

### Parameters

In [88]:
arr_batchs = np.array([1, 2, 3])
dep_batchs = np.array([1, 2, 3, 4, 5])
nservs = np.array([1, 2, 3, 4, 5])
#tmaxs = [100, 500, 1000, 2000, 5000, 10000, 15000, 20000]
tmaxs = [5000]

rr_mxm1 = np.array([0.1, 0.2, 0.3]) # rates ratio: lambda / mu
rr_mmx1 = np.array([0.7, 0.8, 0.9]) # rates ratio: lambda / mu
rr_mmc = np.array([0.7, 0.8, 0.9]) # rates ratio: lambda / mu

lambdas = np.repeat(1.0, 3) # lambda can always be taken as 1
mus_mxm1 = lambdas / rr_mxm1
mus_mmx1 = lambdas / rr_mmx1
mus_mmc = lambdas / rr_mmc 
rep_mc = 10

### Simulating Mx-M-1

In [89]:
# sim_MxM1 = pd.DataFrame()

# for batch in arr_batchs:
#     sim = tab_mc_queue(
#         lambdas = lambdas,
#         mus = mus_mxm1,
#         tmaxs = tmaxs,
#         farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
#         fdep = lambda mu: lambda: np.random.exponential(1/mu),
#         nserv = 1,
#         farr_batch = lambda: batch,
#         fdep_batch = lambda: 1,
#         rep_mc = rep_mc,
#     )

#     sim["batch"] = batch
#     sim_MxM1 = pd.concat([sim_MxM1, sim], ignore_index=True)

In [90]:
sim_MxM1 = pd.DataFrame()

for batch in arr_batchs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mxm1,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = 1,
        farr_batch = lambda: batch,
        fdep_batch = lambda: 1,
        rep_mc = rep_mc,
    )

    sim["batch"] = batch
    metrics = sim.apply(lambda row: mxm1_metrics(row['lambda'], row['mu'], int(row['batch'])), axis=1)
    sim['Wq'] = [t['Wq'] for t in metrics]
    sim['Lq'] = [t['Lq'] for t in metrics]
    sim['Rho'] = [t['Rho'] for t in metrics]
    
    sim_MxM1 = pd.concat([sim_MxM1, sim], ignore_index=True)

100.0%

In [92]:
sim_MxM1[sim_MxM1["tmax"] == 5000].drop(columns=['Rho_var', 'Lq_var', 'Wq_var']).head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,batch,Wq,Lq,Rho
0,1.0,10.000000,5000,0.011358,0.011311,0.099081,1,0.011111,0.011111,0.1
1,1.0,5.000000,5000,0.050233,0.050438,0.201373,1,0.050000,0.050000,0.2
2,1.0,3.333333,5000,0.125960,0.125975,0.299553,1,0.128571,0.128571,0.3
3,1.0,10.000000,5000,0.088616,0.176838,0.199362,2,0.087500,0.175000,0.2
4,1.0,5.000000,5000,0.301939,0.606151,0.402032,2,0.300000,0.600000,0.4
5,1.0,3.333333,5000,0.850045,1.707813,0.603189,2,0.825000,1.650000,0.6
6,1.0,10.000000,5000,0.185162,0.553949,0.298539,3,0.185714,0.557143,0.3
7,1.0,5.000000,5000,0.798380,2.405780,0.602297,3,0.800000,2.400000,0.6
8,1.0,3.333333,5000,5.784599,17.398568,0.900244,3,5.700000,17.100000,0.9


### Simulating M-Mx-1

In [ ]:
# sim_MMx1 = pd.DataFrame()

# for batch in dep_batchs:
#     sim = tab_mc_queue(
#         lambdas = lambdas,
#         mus = mus_mmx1,
#         tmaxs = tmaxs,
#         farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
#         fdep = lambda mu: lambda: np.random.exponential(1/mu),
#         nserv = 1,
#         farr_batch = lambda: 1,
#         fdep_batch = lambda: batch,
#         rep_mc = rep_mc,
#     )
#     sim["batch"] = batch
#     sim_MMx1 = pd.concat([sim_MMx1, sim], ignore_index=True)

100.0%

In [93]:
sim_MMx1 = pd.DataFrame()

for batch in dep_batchs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mmx1,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda), # so we have a function for each lambda
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = 1,
        farr_batch = lambda: 1,
        fdep_batch = lambda: batch,
        rep_mc = rep_mc,
    )
    sim["batch"] = batch
    metrics = sim.apply(lambda row: mmx1_metrics(row['lambda'], row['mu'], int(row['batch'])), axis=1)
    sim['Wq'] = [t['Wq'] for t in metrics]
    sim['Lq'] = [t['Lq'] for t in metrics]
    sim['Rho'] = [t['Rho'] for t in metrics]

    sim_MMx1 = pd.concat([sim_MMx1, sim], ignore_index=True)

100.0%

In [94]:
sim_MMx1[sim_MMx1["tmax"] == 5000].drop(columns=['Rho_var', 'Lq_var', 'Wq_var']).head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,batch,Wq,Lq,Rho
0,1.0,1.428571,5000,1.625743,1.625855,0.697122,1,1.633333,1.633333,0.700000
1,1.0,1.250000,5000,3.156721,3.158662,0.798983,1,3.200000,3.200000,0.800000
2,1.0,1.111111,5000,7.991527,8.018158,0.903023,1,8.100000,8.100000,0.900000
3,1.0,1.428571,5000,0.521765,0.521572,0.570638,2,0.203600,0.203600,0.474679
4,1.0,1.250000,5000,0.679501,0.678737,0.625386,2,0.303913,0.303913,0.524695
5,1.0,1.111111,5000,0.885544,0.881999,0.672793,2,0.438528,0.438528,0.572381
6,1.0,1.428571,5000,0.417072,0.415798,0.549149,3,0.061528,0.061528,0.432311
7,1.0,1.250000,5000,0.529974,0.527505,0.596740,3,0.094024,0.094024,0.472024
8,1.0,1.111111,5000,0.655088,0.652919,0.643778,3,0.136729,0.136729,0.509017
9,1.0,1.428571,5000,0.392842,0.392086,0.544313,4,0.022349,0.022349,0.419397


### Simulating M-M-c

In [9]:
sim_MMc = pd.DataFrame()

for nserv in nservs:
    sim = tab_mc_queue(
        lambdas = lambdas,
        mus = mus_mmc,
        tmaxs = tmaxs,
        farr = lambda lmbda: lambda: np.random.exponential(1/lmbda),
        fdep = lambda mu: lambda: np.random.exponential(1/mu),
        nserv = nserv,
        farr_batch = lambda: 1,
        fdep_batch = lambda: 1,
        rep_mc = rep_mc,
    )

    sim["nserv"] = nserv
    sim_MMc = pd.concat([sim_MMc, sim], ignore_index=True)

100.0%

In [ ]:
sim_MMc[sim_MMc["nserv"] == 3].drop(columns=['Rho_var', 'Lq_var', 'Wq_var']).head(10)

,lambda,mu,tmax,Wq_mean,Lq_mean,Rho_mean,Wq_var,Lq_var,Rho_var,nserv
48,1.0,1.428571,100,0.011634,0.011809,0.230324,0.000187,0.000194,0.000819,3
49,1.0,1.428571,500,0.010433,0.010425,0.230615,0.000030,0.000030,0.000165,3
50,1.0,1.428571,1000,0.011426,0.011509,0.234405,0.000025,0.000027,0.000102,3
51,1.0,1.428571,2000,0.011469,0.011487,0.233408,0.000011,0.000011,0.000055,3
52,1.0,1.428571,5000,0.011569,0.011555,0.232469,0.000006,0.000006,0.000022,3
53,1.0,1.428571,10000,0.011364,0.011364,0.233339,0.000002,0.000003,0.000013,3
54,1.0,1.428571,15000,0.011338,0.011343,0.233474,0.000002,0.000002,0.000008,3
55,1.0,1.428571,20000,0.011351,0.011345,0.232896,0.000001,0.000001,0.000006,3
56,1.0,1.250000,100,0.018534,0.018848,0.264371,0.000349,0.000371,0.001288,3
57,1.0,1.250000,500,0.018901,0.019151,0.268597,0.000070,0.000077,0.000295,3


### Save results and clear variables

In [11]:
filename = f"sim_{date.today()}.pkl"

results = {
    'lambdas': lambdas,
    'rr_mxm1': mus_mxm1,
    'rr_mmx1': mus_mmx1,
    'rr_mmc': mus_mmc,

    'lambdas': lambdas,
    'mus_mxm1': mus_mxm1,
    'mus_mmx1': mus_mmx1,
    'mus_mmc': mus_mmc,

    'arr_batch': arr_batchs,
    'dep_batch': dep_batchs,
    'nservs': nservs,
    'tmaxs': tmaxs,
    'rep_mc': rep_mc,
    'sim_MxM1': sim_MxM1,
    'sim_MMx1': sim_MMx1,
    'sim_MMc': sim_MMc,
}

with open(filename, 'wb') as f:
    pickle.dump(results, f)

del rr_mxm1, rr_mmx1, rr_mmc, lambdas, mus_mxm1, mus_mmx1, mus_mmc
del tmaxs, rep_mc, arr_batchs, dep_batchs, filename, results

## Testing queue generator
---

### M-M-c test

Theoretical metrics

In [ ]:
def mmc_metrics(lmbda, mu, c):
    rho = lmbda / (c * mu)
    s   = sum((lmbda / mu)**n / math.factorial(n) for n in range(c))
    t   = (lmbda / mu)**c / (math.factorial(c) * (1 - rho))
    P0  = 1 / (s + t)
    Pw  = t * P0
    Wq  = Pw / (c * mu - lmbda)
    W   = Wq + 1 / mu
    Lq  = lmbda * Wq
    L   = lmbda * W
    
    return {
        'rho': rho,
        'Pw': Pw,
        'Wq': Wq,
        'W': W,
        'Lq': Lq,
        'L': L
    }

In [ ]:
nserv  = 3
rho    = 0.5
lmbda  = 2.0
mu     = lmbda / (nserv * rho)

rho, Pw, Wq, W, Lq, L = mmc_metrics(lmbda, mu, nserv)

print("Rho:", round(rho, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))
print("Lq :", round(Lq, 4))
print("L  :", round(L, 4))
print("Pw :", round(Pw, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda)
fdep = lambda: rnd.expovariate(mu)

q = Queue(nserv=nserv, farr=farr, fdep=fdep, tmax=10000)

q.run()
q.print_metrics()

### Mx-M-1 test

Theoretical metrics

In [ ]:
def mxm1_metrics(lmbda, batch_size, mu_global):
    lmbda_global = lmbda * batch_size
    rho       = lmbda_global / mu_global
    EX2       = batch_size**2
    L         = (rho + (lmbda / mu_global) * EX2) / (2 * (1 - rho))
    W         = L / lmbda_global
    Wq        = W - 1 / mu_global
    Lq        = lmbda_global * Wq
    return {
        'lambda_global': float(lmbda_global),
        'rho': float(rho),
        'L': float(L),
        'Lq': float(Lq),
        'W': float(W),
        'Wq': float(Wq)
    }

In [ ]:
lmbda_val = 0.2
batch     = 5
mu_val    = 2.0

lmbda_eff, rho, L, Lq, W, Wq = mxm1_metrics(lmbda_val, batch, mu_val)

print("rho       :", round(rho, 4))
print("L         :", round(L, 4))
print("Lq        :", round(Lq, 4))
print("W         :", round(W, 4))
print("Wq        :", round(Wq, 4))
print("lambda_eff:", round(lmbda_eff, 4))

Simulated metrics

In [ ]:
farr = lambda: rnd.expovariate(lmbda_val)
fdep = lambda: rnd.expovariate(mu_val)
farr_batch=lambda: batch
fdep_batch=lambda: 1

q = Queue(nserv=1, farr=farr, fdep=fdep, farr_batch=farr_batch, fdep_batch=fdep_batch, tmax=100000)

q.run()
q.print_metrics()

### M-G-1 test

Theoretical metrics

In [ ]:
def mg1_metrics(lmbda, ES, VarS):
    rho = lmbda * ES
    Cv2 = VarS / (ES**2)

    Lq  = ((1 + Cv2) / 2) * (rho**2 / (1 - rho))
    W   = ((1 + Cv2) / 2) * (rho / (1/ES - lmbda)) + ES
    L   = Lq + rho
    Wq  = W - ES

    return rho, L, Lq, W, Wq

lmbda = 0.8
a, b  = 0.7, 1.7

ES    = (a + b) / 2
VarS  = (b - a)**2 / 12

rho, L, Lq, W, Wq = mg1_metrics(lmbda, ES, VarS)

print("rho:", round(rho, 4))
print("L  :", round(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

In [ ]:
(L, 4))
print("Lq :", round(Lq, 4))
print("W  :", round(W, 4))
print("Wq :", round(Wq, 4))

Simulated metrics

In [ ]:
q = Queue(
    farr = lambda: rnd.expovariate(lmbda),
    fdep = lambda: rnd.uniform(a, b),
    tmax = 200000
)

q.run()
q.print_metrics()